In [188]:
import os
import pandas as pd

from loguru import logger
from pathlib import Path

In [189]:
def list_files_in_folder(folder_path, include_subfolders=False):
    """
    Lists all files in the given folder.
    
    :param folder_path: Path to the folder to scan.
    :param include_subfolders: If True, will also scan subfolders.
    """
   
    try:
        folder = Path(folder_path)
        files = []

        if not folder.exists():
            print(f"Error: The folder '{folder_path}' does not exist.")
            return

        if include_subfolders:
            # Recursively list all files
            for file_path in folder.rglob("*"):
                if file_path.is_file():
                    files.append(file_path)
        else:
            # Only list files in the top-level folder
            for file_path in folder.iterdir():
                if file_path.is_file():
                    files.append(file_path)
        return files

    except PermissionError:
        print("Error: Permission denied while accessing the folder.")
    except Exception as e:
        print(f"Unexpected error: {e}")

In [190]:
def drop_na(df):
    """Function to Drop any NA rows and return the number dropped"""
    size = len(df)
    df.dropna(inplace=True)
    logger.info(f"{size - len(df)} row(s) dropped")
    return df

In [191]:
def clear_whitespace(df):
    """Function to clear whitespace from string columns in dataframe"""
    for i in df.columns:
        if df[i].dtype == 'object':
            df[i] = df[i].map(str.strip)
        else:
            pass
    return df

In [192]:
def float_to_int(df, columns):
    df[columns] = df[columns].apply(pd.to_numeric, errors='coerce').round().astype('Int64')
    return df

In [193]:
def dates(df, columns):

    df[columns] = df[columns].apply(pd.to_datetime, dayfirst=True,
            errors="coerce",format="mixed",)
    return df

In [194]:
customer = {"Int":["Customer ID"]}
books = {"Int":["Id", "Customer ID"],
        "Date":["Book checkout", "Book Returned"]}

In [195]:
mapping = books
mapping["Date"]

['Book checkout', 'Book Returned']

In [196]:
folder = "data_in"
files = list_files_in_folder(folder, include_subfolders=False)
for file in files:
    if "book" in str(file).lower():
        mapping = books
        output = "data_out/books.csv"
    else:
        mapping = customer
        output = "data_out/customer.csv"

    df = pd.read_csv(file)
    df = drop_na(df)
    if "Int" in mapping.keys():
        df = float_to_int(df,mapping["Int"])
    df = clear_whitespace(df)
    if "Date" in mapping.keys():
        df = dates(df, mapping["Date"])
    try:
        df.to_csv(output, index=False, encoding="utf-8")
        logger.info(f"{output} written successfully")
    except Exception as e:
        logger.info(f"Unable to write due to {e}")

2026-06-01 15:33:55.815 | INFO     | __main__:drop_na:5 - 94 row(s) dropped
2026-06-01 15:33:55.831 | INFO     | __main__:<module>:20 - data_out/books.csv written successfully
2026-06-01 15:33:55.836 | INFO     | __main__:drop_na:5 - 1 row(s) dropped
2026-06-01 15:33:55.843 | INFO     | __main__:<module>:20 - data_out/customer.csv written successfully
